# CLIK Workshop 2 - End-to-End ML di Snowflake## Use case: Probability of Default (PD)**Alur:**1. Setup & Snowpark session2. Load data (1 juta baris, ~196 fitur)3. Feature engineering & split4. Logistic Regression (stepwise)5. XGBoost6. LightGBM7. Evaluasi (AUC, Gini, KS, ROC)8. Register ke Model Registry9. Test inference> **Scaling:** dataset sudah 1jt baris. Untuk 5jt+, naikkan warehouse size & gunakan distributed training.

## 1. Setup

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom snowflake.snowpark.context import get_active_sessionsession = get_active_session()session.use_database('CLIK_WORKSHOP2')session.use_schema('PUBLIC')session.use_warehouse('GEN2_SMALL')print('Connected:', session.get_current_database(), session.get_current_schema())

## 2. Load & Eksplorasi

In [ ]:
TARGET = 'DEFAULT_FLAG'SAMPLE_ROWS = 300_000LR_SAMPLE = 120_000LEAK_COLS = ['SUBJECT_ID', 'PD_TRUE_PROB']snow_df = session.table('SUBJECT_FEATURES')print(f'Total rows: {snow_df.count():,}')snow_df.select('DEFAULT_FLAG').group_by('DEFAULT_FLAG').count().show()

In [ ]:
pdf = snow_df.sample(n=SAMPLE_ROWS).to_pandas()print('Sample shape:', pdf.shape)print('Default rate:', round(pdf[TARGET].mean(), 4))CAT_COLS = ['GENDER', 'EMPLOYMENT_TYPE', 'EDUCATION', 'REGION_CODE']DROP = LEAK_COLS + [TARGET]NUM_COLS = [c for c in pdf.columns if c not in CAT_COLS + DROP]print(f'Numeric: {len(NUM_COLS)} | Categorical: {len(CAT_COLS)}')

## 3. Feature Engineering & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.compose import ColumnTransformerfrom sklearn.preprocessing import OneHotEncoder, StandardScalerfrom sklearn.pipeline import PipelineFEATURES = NUM_COLS + CAT_COLSX = pdf[FEATURES].copy()y = pdf[TARGET].astype(int)X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.25, random_state=42, stratify=y)print('Train:', X_train.shape, '| Test:', X_test.shape)preprocess = ColumnTransformer([    ('num', StandardScaler(), NUM_COLS),    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),])

## 4. Model 1 - Logistic Regression (Stepwise Forward Selection)Ranking fitur by univariate AUC, forward selection max 25 fitur.

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import roc_auc_scoreXtr_lr = X_train.sample(min(LR_SAMPLE, len(X_train)), random_state=1)ytr_lr = y_train.loc[Xtr_lr.index]uni = {}for c in NUM_COLS:    try:        uni[c] = abs(roc_auc_score(ytr_lr, Xtr_lr[c].fillna(0)) - 0.5)    except:        uni[c] = 0ranked = sorted(uni, key=uni.get, reverse=True)candidate_pool = ranked[:40] + CAT_COLSprint('Top 10 features:', ranked[:10])

In [ ]:
from sklearn.model_selection import train_test_split as ttsXa, Xv, ya, yv = tts(Xtr_lr, ytr_lr, test_size=0.3, random_state=7, stratify=ytr_lr)def fit_auc(cols):    num = [c for c in cols if c in NUM_COLS]    cat = [c for c in cols if c in CAT_COLS]    ct = ColumnTransformer(        [('num', StandardScaler(), num)] +        ([('cat', OneHotEncoder(handle_unknown='ignore'), cat)] if cat else []))    pipe = Pipeline([('prep', ct), ('lr', LogisticRegression(max_iter=200, C=1.0))])    pipe.fit(Xa[cols], ya)    return roc_auc_score(yv, pipe.predict_proba(Xv[cols])[:, 1]), pipeselected, best_auc = [], 0.5for _ in range(25):    trial_best, trial_col = best_auc, None    for c in candidate_pool:        if c in selected: continue        auc, _ = fit_auc(selected + [c])        if auc > trial_best + 1e-4:            trial_best, trial_col = auc, c    if trial_col is None: break    selected.append(trial_col); best_auc = trial_best    print(f'+ {trial_col:30s} -> AUC {best_auc:.4f} ({len(selected)} feats)')print('\nSelected:', selected)

In [ ]:
num_sel = [c for c in selected if c in NUM_COLS]cat_sel = [c for c in selected if c in CAT_COLS]lr_prep = ColumnTransformer(    [('num', StandardScaler(), num_sel)] +    ([('cat', OneHotEncoder(handle_unknown='ignore'), cat_sel)] if cat_sel else []))lr_model = Pipeline([('prep', lr_prep), ('lr', LogisticRegression(max_iter=500, C=1.0))])lr_model.fit(X_train[selected], y_train)lr_pred = lr_model.predict_proba(X_test[selected])[:, 1]lr_auc = roc_auc_score(y_test, lr_pred)print(f'Logistic Regression (stepwise) - AUC: {lr_auc:.4f} | Gini: {2*lr_auc-1:.4f}')

## 5. Model 2 - XGBoost

In [ ]:
import xgboost as xgbxgb_clf = xgb.XGBClassifier(    n_estimators=300, max_depth=6, learning_rate=0.1,    subsample=0.8, colsample_bytree=0.8, eval_metric='auc',    scale_pos_weight=(1-y_train.mean())/y_train.mean(), n_jobs=-1, random_state=42)xgb_model = Pipeline([('prep', preprocess), ('xgb', xgb_clf)])xgb_model.fit(X_train, y_train)xgb_pred = xgb_model.predict_proba(X_test)[:, 1]xgb_auc = roc_auc_score(y_test, xgb_pred)print(f'XGBoost - AUC: {xgb_auc:.4f} | Gini: {2*xgb_auc-1:.4f}')

## 6. Model 3 - LightGBM

In [ ]:
import lightgbm as lgblgb_clf = lgb.LGBMClassifier(    n_estimators=400, num_leaves=48, learning_rate=0.05,    subsample=0.8, colsample_bytree=0.8, class_weight='balanced',    n_jobs=-1, random_state=42, verbose=-1)lgb_model = Pipeline([('prep', preprocess), ('lgbm', lgb_clf)])lgb_model.fit(X_train, y_train)lgb_pred = lgb_model.predict_proba(X_test)[:, 1]lgb_auc = roc_auc_score(y_test, lgb_pred)print(f'LightGBM - AUC: {lgb_auc:.4f} | Gini: {2*lgb_auc-1:.4f}')

## 7. Model Evaluation - AUC, Gini, KS, ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, classification_reportdef ks_stat(y_true, y_score):    fpr, tpr, _ = roc_curve(y_true, y_score)    return float(np.max(tpr - fpr))results = pd.DataFrame({    'Model': ['LogReg (stepwise)', 'XGBoost', 'LightGBM'],    'AUC':  [lr_auc, xgb_auc, lgb_auc],    'Gini': [2*lr_auc-1, 2*xgb_auc-1, 2*lgb_auc-1],    'KS':   [ks_stat(y_test, lr_pred), ks_stat(y_test, xgb_pred), ks_stat(y_test, lgb_pred)],}).sort_values('AUC', ascending=False).reset_index(drop=True)display(results)

In [ ]:
plt.figure(figsize=(7,5))for name, pred in [('LogReg', lr_pred), ('XGBoost', xgb_pred), ('LightGBM', lgb_pred)]:    fpr, tpr, _ = roc_curve(y_test, pred)    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test,pred):.3f})')plt.plot([0,1],[0,1],'k--',alpha=0.4)plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curves - PD Models')plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
best_name = results.iloc[0]['Model']best_pred = {'LogReg (stepwise)': lr_pred, 'XGBoost': xgb_pred, 'LightGBM': lgb_pred}[best_name]cm = confusion_matrix(y_test, (best_pred >= 0.5).astype(int))print(f'Best: {best_name}')print(f'Confusion Matrix (threshold=0.5):\n{cm}')print(classification_report(y_test, (best_pred >= 0.5).astype(int), digits=3))

## 8. Register Best Model ke Snowflake Model RegistryPipeline lengkap (preprocess + classifier) -> inference terima kolom mentah.

In [ ]:
from snowflake.ml.registry import Registrybest_model_obj = {'LogReg (stepwise)': lr_model, 'XGBoost': xgb_model, 'LightGBM': lgb_model}[best_name]best_features = selected if best_name.startswith('LogReg') else FEATURESreg = Registry(session=session, database_name='CLIK_WORKSHOP2', schema_name='PUBLIC')sample_input = X_train[best_features].head(50)mv = reg.log_model(    model=best_model_obj,    model_name='CLIK_PD_MODEL',    version_name='V1',    sample_input_data=sample_input,    comment=f'PD model ({best_name}). AUC={results.iloc[0]["AUC"]:.4f}',    metrics={'test_auc': float(results.iloc[0]['AUC']),             'test_gini': float(results.iloc[0]['Gini']),             'test_ks': float(results.iloc[0]['KS'])},    conda_dependencies=['scikit-learn', 'xgboost', 'lightgbm', 'pandas', 'numpy'],)print('Registered:', mv.model_name, mv.version_name)

In [ ]:
m = reg.get_model('CLIK_PD_MODEL')m.default = 'V1'reg.show_models()

## 9. Test Inference dari Registry (Warehouse)

In [ ]:
test_sdf = session.table('SUBJECT_FEATURES').limit(10)pred_sdf = mv.run(test_sdf, function_name='predict_proba')pred_sdf.show()

## Selesai!Model `CLIK_PD_MODEL` v1 sudah di registry. Lanjut ke:- **04a** - Batch scoring (warehouse)- **04b** - Real-time inference (SPCS model serving)